# 02 Model Grid Experiment

Title-only logistic regression grid experiment across feature representations, feature budgets, regularization penalties, and C values.


In [ ]:
# Keep paths stable whether this notebook is run from the repo root or from notebooks/.
from pathlib import Path
import os
import sys

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

PROJECT_ROOT = Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


In [3]:
# Imports
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from performance_landscapes.data import (
    balanced_subset,
    clean_title_rows,
    load_raw_news_data,
    split_title_data,
)
from performance_landscapes.modeling import (
    C_VALUES,
    FEATURE_CONFIGS,
    MAX_FEATURES_VALUES,
    PENALTIES,
    run_grid_experiment,
)
from performance_landscapes.paths import PROCESSED_DATA_DIR

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12


In [4]:
# Load and combine data
# Labels: fake = 1, real = 0
df = load_raw_news_data()

print("Combined shape:", df.shape)
print(df["label"].value_counts())


Combined shape: (44898, 5)
label
1    23481
0    21417
Name: count, dtype: int64


In [5]:
# Keep title only
df = clean_title_rows(df)

print("Shape after removing empty titles:", df.shape)


Shape after removing empty titles: (44898, 5)


In [6]:
# Optional subset for speed
subset_size = 10000

df = balanced_subset(df, subset_size=subset_size, random_state=42)

print("Shape after optional subsetting:", df.shape)
print(df["label"].value_counts())


Shape after optional subsetting: (10000, 5)
label
1    5000
0    5000
Name: count, dtype: int64


In [7]:
# Train/test split
X_train, X_test, y_train, y_test = split_title_data(
    df,
    test_size=0.2,
    random_state=42,
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))


Train size: 8000
Test size: 2000


In [8]:
# Experiment settings
C_values = C_VALUES
max_features_values = MAX_FEATURES_VALUES
feature_configs = FEATURE_CONFIGS
penalties = PENALTIES


In [9]:
# Helper functions are imported from performance_landscapes.modeling.


In [10]:
# Run experiments
results_df = run_grid_experiment(
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    feature_configs=feature_configs,
    max_features_values=max_features_values,
    C_values=C_values,
    penalties=penalties,
)

print("\nResults preview:")
print(results_df.head())

print("\nTop rows by Test F1:")
print(results_df.sort_values(by="Test F1", ascending=False).head(20))



Running feature setup: BoW Unigram
  max_features = 100, train matrix shape = (8000, 100)
  max_features = 506, train matrix shape = (8000, 506)
  max_features = 912, train matrix shape = (8000, 912)
  max_features = 1318, train matrix shape = (8000, 1318)
  max_features = 1724, train matrix shape = (8000, 1724)
  max_features = 2130, train matrix shape = (8000, 2130)
  max_features = 2536, train matrix shape = (8000, 2536)
  max_features = 2942, train matrix shape = (8000, 2942)
  max_features = 3348, train matrix shape = (8000, 3348)
  max_features = 3754, train matrix shape = (8000, 3754)
  max_features = 4160, train matrix shape = (8000, 4160)
  max_features = 4566, train matrix shape = (8000, 4566)
  max_features = 4972, train matrix shape = (8000, 4972)
  max_features = 5378, train matrix shape = (8000, 5378)
  max_features = 5784, train matrix shape = (8000, 5784)
  max_features = 6190, train matrix shape = (8000, 6190)
  max_features = 6596, train matrix shape = (8000, 6215)
 

In [16]:
# Save results
results_df.to_csv("fake_news_title_only_logistic_results.csv", index=False)

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
processed_results_path = PROCESSED_DATA_DIR / "landscape_results.csv"
results_df.to_csv(processed_results_path, index=False)

print("Saved results to fake_news_title_only_logistic_results.csv")


Saved results to fake_news_title_only_logistic_results.csv


In [ ]:
# =========================
# 10. Summary tables
# =========================
best_f1 = results_df.sort_values(by="Test F1", ascending=False).head(15)
print("\nBest settings by Test F1:")
print(best_f1[[
    "Feature", "Penalty", "max_features", "C",
    "Test Accuracy", "Test F1", "Test AUC",
    "F1 Gap", "Runtime Seconds"
]])

best_auc = results_df.sort_values(by="Test AUC", ascending=False).head(15)
print("\nBest settings by Test AUC:")
print(best_auc[[
    "Feature", "Penalty", "max_features", "C",
    "Test Accuracy", "Test F1", "Test AUC",
    "AUC Gap", "Runtime Seconds"
]])

summary_table = (
    results_df.groupby(["Feature", "Penalty"])[[
        "Test Accuracy", "Test F1", "Test AUC", "Runtime Seconds", "F1 Gap"
    ]]
    .mean()
    .sort_values(by="Test F1", ascending=False)
)

print("\nAverage performance by feature configuration and penalty:")
print(summary_table)


Best settings by Test F1:
                 Feature  max_features Penalty          C  Test Accuracy  \
16077     TF-IDF Unigram          4160      l2   4.094915         0.9350   
16177     TF-IDF Unigram          4566      l2   4.094915         0.9350   
16078     TF-IDF Unigram          4160      l2   7.196857         0.9345   
15978     TF-IDF Unigram          3754      l2   7.196857         0.9340   
21677  TF-IDF Uni+Bigram          6596      l2   4.094915         0.9340   
21977  TF-IDF Uni+Bigram          7814      l2   4.094915         0.9335   
15979     TF-IDF Unigram          3754      l2  12.648552         0.9335   
21777  TF-IDF Uni+Bigram          7002      l2   4.094915         0.9335   
15977     TF-IDF Unigram          3754      l2   4.094915         0.9335   
21877  TF-IDF Uni+Bigram          7408      l2   4.094915         0.9330   
21477  TF-IDF Uni+Bigram          5784      l2   4.094915         0.9330   
19577     TF-IDF Unigram         18370      l2   4.094915    